In [23]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.metrics import r2_score
from transformers import AutoTokenizer, AutoModelForMaskedLM , AutoModel


In [24]:
folder=os.listdir('data')
train_file=folder[1]
test_file=folder[0]
val_file=folder[2]


In [25]:
def load_data(file_path):
    with open(os.path.join('data',file_path),'r') as f:
        data=pd.read_csv(f)
    return data

In [26]:
data_train=load_data(train_file)
data_test=load_data(test_file)
data_val=load_data(val_file)

data_train['Review'].head()

0    nice hotel expensive parking got good deal sta...
1    ok nothing special charge diamond member hilto...
2    nice rooms not 4* experience hotel monaco seat...
3    unique, great stay, wonderful time hotel monac...
4    great stay great stay, went seahawk game aweso...
Name: Review, dtype: object

In [27]:
# %%
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128, # Adjust based on your GPU memory
        return_tensors="pt"
    )

# Prepare labels (Subtract 1 if your ratings are 1-5 to get 0-4 range)
def prepare_labels(df):
    return torch.tensor(df['Rating'].values - 1, dtype=torch.long)

train_labels = prepare_labels(data_train)
val_labels = prepare_labels(data_val)
test_labels = prepare_labels(data_test)

In [ ]:
""" # %%
class RegressionModel(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # CORN Head: K-1 binary classifiers
        self.corn_head = nn.Linear(self.bert.config.hidden_size, num_classes - 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # ModernBERT uses the first token [CLS] for global representation
        cls_embedding = outputs.last_hidden_state[:, 0, :] 
        logits = self.corn_head(cls_embedding)
        return logits"""

In [37]:
class RegressionModel(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.head = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, num_classes - 1)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Mean Pooling : On fait la moyenne de tous les tokens au lieu de prendre juste le [CLS]
        # Cela aide pour les avis mixtes (ex: échantillon 465)
        mask = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        return self.head(mean_pooled)

In [ ]:
# %%
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_classes = 5
model = RegressionModel(model_id, num_classes=num_classes).to(device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-6)

# CORN requires converting the rank to a set of binary labels
def label_to_corn(labels, num_classes):
    # labels: [batch_size], range [0, num_classes-1]
    sets = []
    for i in range(num_classes - 1):
        sets.append(labels > i)
    return torch.stack(sets, dim=1).float()

batch_size = 16 # BERT is memory intensive; 256 is likely too high for a standard GPU
epochs = 4
all_train_loss = []
all_val_loss = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    # Shuffle indices
    indices = np.arange(len(data_train))
    np.random.shuffle(indices)
    
    for start in range(0, len(indices), batch_size):
        optimizer.zero_grad()
        
        batch_idx = indices[start:start + batch_size]
        batch_text = data_train['Review'].iloc[batch_idx]
        
        inputs = tokenize_data(batch_text).to(device)
        targets = train_labels[batch_idx].to(device)
        corn_targets = label_to_corn(targets, num_classes).to(device)

        logits = model(**inputs)
        
        # Binary Cross Entropy with Logits for CORN
        loss = nn.BCEWithLogitsLoss()(logits, corn_targets)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(data_train) / batch_size)
    all_train_loss.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for start in range(0, len(data_val), batch_size):
            batch_text = data_val['Review'].iloc[start:start+batch_size]
            inputs = tokenize_data(batch_text).to(device)
            targets = val_labels[start:start+batch_size].to(device)
            corn_targets = label_to_corn(targets, num_classes).to(device)
            
            logits = model(**inputs)
            loss = nn.BCEWithLogitsLoss()(logits, corn_targets)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / (len(data_val) / batch_size)
    all_val_loss.append(avg_val_loss)
    print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f}")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 518.51it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1: Train Loss 0.3578 | Val Loss 0.2762
Epoch 2: Train Loss 0.2547 | Val Loss 0.2399
Epoch 3: Train Loss 0.2275 | Val Loss 0.2258


In [30]:
# %%
def predict_rating(logits):
    # Probs for each conditional step
    probs = torch.sigmoid(logits)
    # Sum the binary decisions: rating = sum(probs > 0.5)
    # We add 1 because our internal labels were 0-4
    return torch.sum(probs > 0.5, dim=1) + 1

model.eval()
all_preds = []
all_true = data_test['Rating'].values

with torch.no_grad():
    for start in range(0, len(data_test), batch_size):
        batch_text = data_test['Review'].iloc[start:start+batch_size]
        inputs = tokenize_data(batch_text).to(device)
        
        logits = model(**inputs)
        preds = predict_rating(logits)
        all_preds.extend(preds.cpu().numpy())

print(f"Final R2 Score: {r2_score(all_true, all_preds):.4f}")

Final R2 Score: 0.7031


In [31]:
from sklearn.metrics import mean_absolute_error

# Inside your test/eval loop:
mae = mean_absolute_error(all_true, all_preds)
print(f"Average Star Error (MAE): {mae:.2f}")

Average Star Error (MAE): 0.36


In [36]:
# Code pour trouver les avis qui font chuter votre score
errors = np.abs(np.array(all_true) - np.array(all_preds))
hard_cases = data_test.iloc[np.where(errors > 1.5)]
print(hard_cases['Review'].head(24))

10     sure read just got yesterday wait post review ...
26     pretty hotel hotel really pretty no doubt girl...
74     n't believe reviews read terror tinytown staff...
80     dirty french press time stayed hotel 1000. pro...
120    cranes close comfort** outside window huge cra...
172    favorite 11 nights waikiki stayed 4 nights.ok ...
309    luxurious bathrooms hotel absolutely beautiful...
310    horrible file lawsuit, extremely disappointed ...
378    2nd stay better, returned hoxton 2nd stay like...
465    lovely hotel concierge complete joke stayed 3 ...
593    great beach wish say resort just returned domi...
604    seccond stay breezes seccond visit breezes pun...
620    concerns not warranted breezes 25 years old ju...
631    great vacation minor drawbacks family visited ...
635    breezes n't breeze arrived breezes july 19 day...
649    awesome time sceptical going breezes reading b...
690    started great quickly went hill scheduled stay...
693    intercontinental century

In [29]:
import time
import random
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score

# --- CONFIGURATION DE LA BATAILLE ---
NUM_TRIALS = 15          # On réduit car BERT est lent
MAX_EPOCHS = 5           # BERT converge très vite (souvent 2-3 époques suffisent)
BATCH_SIZE = 16          # Réduire si OOM (Out of Memory)
num_classes = 5

global_best_r2 = -float('inf')
global_best_model_state = None
global_best_params = {}

print(f"⚔️ DÉBUT DE LA BATTLE ROYALE MODERNBERT")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    
    # 1. PARAMÈTRES ALÉATOIRES
    seed = random.randint(0, 10000)
    # Range spécifique pour BERT (entre 1e-6 et 5e-5)
    lr = 10 ** np.random.uniform(-5.7, -4.3) 
    
    # 2. FIXER LA GRAINE
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU MODÈLE
    model = RegressionModel(model_id, num_classes=num_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    
    best_val_loss_trial = float('inf')
    best_model_trial_state = None 
    
    # --- ENTRAÎNEMENT ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        total_train_loss = 0
        
        # Mélange des données
        indices = np.arange(len(data_train))
        np.random.shuffle(indices)
        
        for start in range(0, len(indices), BATCH_SIZE):
            batch_idx = indices[start:start + BATCH_SIZE]
            inputs = tokenize_data(data_train['Review'].iloc[batch_idx]).to(device)
            targets = train_labels[batch_idx].to(device)
            corn_targets = label_to_corn(targets, num_classes).to(device)

            optimizer.zero_grad()
            logits = model(**inputs)
            loss = loss_fn(logits, corn_targets)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for start in range(0, len(data_val), BATCH_SIZE):
                inputs = tokenize_data(data_val['Review'].iloc[start:start+BATCH_SIZE]).to(device)
                targets = val_labels[start:start+BATCH_SIZE].to(device)
                corn_targets = label_to_corn(targets, num_classes).to(device)
                
                logits = model(**inputs)
                total_val_loss += loss_fn(logits, corn_targets).item()
        
        avg_val_loss = total_val_loss / (len(data_val)/BATCH_SIZE)
        
        if avg_val_loss < best_val_loss_trial:
            best_val_loss_trial = avg_val_loss
            best_model_trial_state = copy.deepcopy(model.state_dict())

    # --- TEST DU CHAMPION LOCAL ---
    model.load_state_dict(best_model_trial_state)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for start in range(0, len(data_test), BATCH_SIZE):
            inputs = tokenize_data(data_test['Review'].iloc[start:start+batch_size]).to(device)
            logits = model(**inputs)
            all_preds.extend(predict_rating(logits).cpu().numpy())
    
    current_test_r2 = r2_score(data_test['Rating'].values, all_preds)
    
    # --- VERDICT ---
    if current_test_r2 > global_best_r2:
        print(f"🔥 [Trial {trial:2d}] NOUVEAU ROI ! R2: {current_test_r2:.4f} | LR: {lr:.6f} | Seed: {seed}")
        global_best_r2 = current_test_r2
        global_best_model_state = best_model_trial_state
        global_best_params = {'seed': seed, 'lr': lr}
    else:
        print(f"   [Trial {trial:2d}] Test R2: {current_test_r2:.4f} (Best: {global_best_r2:.4f})")

    # Nettoyage mémoire pour le prochain combat
    del model
    torch.cuda.empty_cache()

duration = (time.time() - start_time) / 60
print("\n" + "="*60)
print(f"🏁 FINI en {duration:.1f} min. Meilleur R2: {global_best_r2:.4f}")
print(f"⚙️ Paramètres : {global_best_params}")

# Sauvegarde finale
torch.save(global_best_model_state, 'best_modernbert_corn.pth')

⚔️ DÉBUT DE LA BATTLE ROYALE MODERNBERT


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 932.11it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔥 [Trial  1] NOUVEAU ROI ! R2: 0.7244 | LR: 0.000030 | Seed: 8915


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1101.57it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   [Trial  2] Test R2: 0.7031 (Best: 0.7244)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1056.60it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (9 outputs left out)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1048.12it/s, Materializing param=layers.21.mlp_norm.weight]    
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   [Trial  8] Test R2: 0.7095 (Best: 0.7357)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1092.31it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

- 🔥 [Trial  1] NOUVEAU ROI ! R2: 0.7046 | LR: 0.000003 | Seed: 2817
- 🔥 [Trial  3] NOUVEAU ROI ! R2: 0.7053 | LR: 0.000018 | Seed: 422
- 🔥 [Trial  1] NOUVEAU ROI ! R2: 0.7294 | LR: 0.000009 | Seed: 7483
- 🔥 [Trial  3] NOUVEAU ROI ! R2: 0.7357 | LR: 0.000038 | Seed: 2726




In [ ]:
#torch.save(model.state_dict(),'model_v1/model_v5.pth')

In [ ]:
print(f"Mémoire allouée: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Mémoire réservée: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")


Mémoire allouée: 2407.16 MB
Mémoire réservée: 2774.00 MB
